In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.stats import linregress

from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from IPython.display import Audio
import os
from scipy.io.wavfile import write

In [ ]:
CACHE_PATH = '../../data/features_cache/znacajke_cache_10.pkl'
df = pd.read_pickle(CACHE_PATH)
print(f"Znacajke ucitane iz cachea: {CACHE_PATH} ({len(df)} uzoraka)")

X = df.drop(columns=['label'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

In [ ]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

X_tr2, X_val, y_tr2, y_val = train_test_split(X_train, y_train_enc, test_size=0.2, stratify=y_train_enc, random_state=42)
weights_tr2 = compute_sample_weight('balanced', y_tr2)

clf_es = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    early_stopping_rounds=30,
    random_state=42,
    n_jobs=-1,
)
clf_es.fit(
    X_tr2, y_tr2, sample_weight=weights_tr2,
    eval_set=[(X_val, y_val)],
    verbose=False
)

train_acc_es = clf_es.score(X_tr2, y_tr2)
test_acc_es = clf_es.score(X_test, y_test_enc)

print(f"Train accuracy: {train_acc_es:.3f}")
print(f"Test accuracy:  {test_acc_es:.3f}")
print(f"Razlika (train - test): {train_acc_es - test_acc_es:.3f}")


print(f"Zaustavljeno nakon {clf_es.best_iteration} stabala (od maksimalnih 1000 dopustenih)")

y_pred_es = le.inverse_transform(clf_es.predict(X_test))
print(classification_report(y_test, y_pred_es))

cm = confusion_matrix(y_test, y_pred_es)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(cmap='Blues')
plt.show()

## Export u ONNX

Isti pristup kao u `scripts/export_modeli.py`: ONNX converter za XGBoost zahtijeva featuree
bez imena (numpy array, ne DataFrame) i deterministican broj stabala, pa refitamo bez early
stoppinga na tocno onoliko stabala koliko je early stopping odabrao (boosting je deterministican
pa su stabla identicna kao u `clf_es`, samo bez "viska" iza `best_iteration`).

In [ ]:
from onnxmltools.convert import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType as OnnxMlFloatTensorType
import onnxruntime as ort

EXPORT_DIR = '../../modeli_exports'
os.makedirs(EXPORT_DIR, exist_ok=True)

n_features = X_train.shape[1]
xgb_params = dict(
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
    n_jobs=-1,
)

n_best = clf_es.best_iteration + 1
X_tr2_np, X_test_np = X_tr2.to_numpy(), X_test.to_numpy()

xgb_final = XGBClassifier(n_estimators=n_best, **xgb_params)
xgb_final.fit(X_tr2_np, y_tr2, sample_weight=weights_tr2)
print(f"Test accuracy (finalni model, {n_best} stabala): {xgb_final.score(X_test_np, y_test_enc):.3f}")

onnx_model = convert_xgboost(
    xgb_final,
    initial_types=[('input', OnnxMlFloatTensorType([None, n_features]))],
)
onnx_path = os.path.join(EXPORT_DIR, 'xgboost_10_mfcc.onnx')
with open(onnx_path, 'wb') as f:
    f.write(onnx_model.SerializeToString())

# provjera da ONNX daje iste predikcije kao originalni model
sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
input_name = sess.get_inputs()[0].name
onnx_preds = sess.run(None, {input_name: X_test_np.astype(np.float32)})[0]
match = np.mean(np.asarray(onnx_preds) == np.asarray(y_test_enc))
print(f"Podudaranje predikcija ONNX vs original: {match:.4%}")

# mapiranje broj -> klasa i redoslijed znacajki, potrebno za kontroler
with open(os.path.join(EXPORT_DIR, 'xgboost_10_mfcc_label_mapping.txt'), 'w') as f:
    for i, cls in enumerate(le.classes_):
        f.write(f"{i} {cls}\n")

with open(os.path.join(EXPORT_DIR, 'xgboost_10_mfcc_redoslijed_znacajki.txt'), 'w') as f:
    for i, col in enumerate(X.columns):
        f.write(f"{i} {col}\n")

print(f"Model spremljen: {onnx_path}")